# Experiment 01 — Perch v2 Frozen Embeddings + LR Probes

**Approach:** Run frozen Perch v2 ONNX on labeled soundscape windows → extract 1536-d embeddings + mapped species logits → train per-class logistic regression probes via GroupKFold (grouped by soundscape file).

**Metric:** OOF macro AUC + padded cmAP (competition metric approximation)

In [1]:
import gc
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
import onnxruntime as ort
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from scipy.special import expit as sigmoid
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DATA_DIR          = Path("../data")
SOUNDSCAPE_DIR    = DATA_DIR / "train_soundscapes"
PERCH_ONNX_PATH   = Path("../data/models/perch/perch_v2_no_dft.onnx")
PERCH_LABELS_PATH = Path("../data/models/perch_tf/assets/labels.csv")
CACHE_DIR         = Path("../data/cache")
CACHE_DIR.mkdir(exist_ok=True)

SR          = 32_000
WIN_SAMPLES = SR * 5
BATCH_SIZE  = 4
N_SPLITS    = 5
PCA_DIM     = 64
MIN_POS     = 3

print("onnxruntime:", ort.__version__)

onnxruntime: 1.23.2


In [2]:
# ── Taxonomy + Perch label mapping ──────────────────────────────────────────
taxonomy = pd.read_csv(DATA_DIR / "taxonomy.csv")
N_CLASSES = len(taxonomy)
# primary_label can be iNat taxon ID (int) or eBird code (str) — keep as str
taxon_ids     = taxonomy["primary_label"].astype(str).tolist()
taxon_to_idx  = {tid: i for i, tid in enumerate(taxon_ids)}

perch_labels = pd.read_csv(PERCH_LABELS_PATH, header=0)
perch_labels.columns = ["scientific_name"]
perch_labels["bc_index"] = np.arange(len(perch_labels))

mapping     = taxonomy.merge(perch_labels, on="scientific_name", how="left")
mapped_mask = mapping["bc_index"].notna()
print(f"Mapped {mapped_mask.sum()}/{N_CLASSES} competition species → Perch index")

comp_to_bc = {}
for _, row in mapping[mapped_mask].iterrows():
    comp_idx = taxon_to_idx[str(row["primary_label"])]   # str, not int
    comp_to_bc[comp_idx] = int(row["bc_index"])

print(f"Unmapped species: {(~mapped_mask).sum()} (no direct Perch logit)")

Mapped 203/234 competition species → Perch index
Unmapped species: 31 (no direct Perch logit)


In [3]:
# ── Load soundscape labels + build Y ────────────────────────────────────────
labels_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")

def time_to_sec(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

labels_df["start_sec"] = labels_df["start"].apply(time_to_sec)
labels_df = labels_df.reset_index(drop=True)

def parse_labels(label_str):
    vec = np.zeros(N_CLASSES, dtype=np.float32)
    for tid in str(label_str).split(";"):
        tid = tid.strip()
        if tid in taxon_to_idx:
            vec[taxon_to_idx[tid]] = 1.0
    return vec

Y = np.stack(labels_df["primary_label"].apply(parse_labels).values)

print(f"Labeled windows : {len(labels_df)} from {labels_df['filename'].nunique()} files")
print(f"Label matrix    : {Y.shape}")
print(f"Classes w/ >=1 positive: {(Y.sum(axis=0) > 0).sum()}")
print(f"Mean positives per window: {Y.sum(axis=1).mean():.2f}")

Labeled windows : 1478 from 66 files
Label matrix    : (1478, 234)
Classes w/ >=1 positive: 75
Mean positives per window: 4.22


In [4]:
# ── Perch ONNX inference (with disk cache) ───────────────────────────────────
EMB_CACHE  = CACHE_DIR / "exp01_embeddings.npy"
LOG_CACHE  = CACHE_DIR / "exp01_logits.npy"

if EMB_CACHE.exists() and LOG_CACHE.exists():
    print("Loading cached Perch outputs...")
    emb_full    = np.load(EMB_CACHE)
    logits_full = np.load(LOG_CACHE)
else:
    print("Running Perch ONNX inference (caching to disk after)...")
    sess = ort.InferenceSession(str(PERCH_ONNX_PATH), providers=["CPUExecutionProvider"])

    def run_perch(waves):  # (B, 160000) float32 → embeddings, logits
        out = sess.run(["embedding", "label"], {"inputs": waves})
        return out[0], out[1]

    all_embs, all_logits = [], []

    for fname, grp in tqdm(labels_df.groupby("filename"), desc="Files"):
        audio, _ = sf.read(str(SOUNDSCAPE_DIR / fname), dtype="float32", always_2d=False)

        chunks = []
        for _, row in grp.iterrows():
            start = int(row["start_sec"]) * SR
            chunk = audio[start : start + WIN_SAMPLES]
            if len(chunk) < WIN_SAMPLES:
                chunk = np.pad(chunk, (0, WIN_SAMPLES - len(chunk)))
            chunks.append(chunk.astype(np.float32))

        embs, logits = [], []
        for i in range(0, len(chunks), BATCH_SIZE):
            b = np.stack(chunks[i : i + BATCH_SIZE])
            e, l = run_perch(b)
            embs.append(e)
            logits.append(l)

        all_embs.append(np.concatenate(embs,   axis=0))
        all_logits.append(np.concatenate(logits, axis=0))
        gc.collect()

    emb_full    = np.concatenate(all_embs,   axis=0).astype(np.float32)
    logits_full = np.concatenate(all_logits, axis=0).astype(np.float32)
    np.save(EMB_CACHE,  emb_full)
    np.save(LOG_CACHE,  logits_full)
    print("Cached.")

print(f"Embeddings: {emb_full.shape}  |  Logits: {logits_full.shape}")

Running Perch ONNX inference (caching to disk after)...


Files:   0%|          | 0/66 [00:00<?, ?it/s]

Cached.
Embeddings: (1478, 1536)  |  Logits: (1478, 14795)


In [5]:
# ── Baseline: raw Perch logits mapped to competition species ─────────────────
scores_raw = np.zeros((len(labels_df), N_CLASSES), dtype=np.float32)
for comp_idx, bc_idx in comp_to_bc.items():
    scores_raw[:, comp_idx] = logits_full[:, bc_idx]

def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return np.mean(aps)

def macro_auc(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        if 0 < y_true[:, c].sum() < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return np.mean(aucs) if aucs else 0.0, len(aucs)

scores_sig  = sigmoid(scores_raw)
auc_raw, n  = macro_auc(Y, scores_sig)
cmap_raw    = padded_cmap(Y, scores_sig)
print(f"Raw Perch logits  →  Macro AUC: {auc_raw:.4f} ({n} classes)  |  Padded cmAP: {cmap_raw:.4f}")

Raw Perch logits  →  Macro AUC: 0.4869 (75 classes)  |  Padded cmAP: 0.7736


In [6]:
# ── OOF LR probes on PCA(embeddings) + raw Perch scores ─────────────────────
groups    = labels_df["filename"].values
gkf       = GroupKFold(n_splits=N_SPLITS)
oof_preds = sigmoid(scores_raw).copy()   # fallback = raw Perch score

for fold, (tr, va) in enumerate(gkf.split(emb_full, groups=groups)):
    scaler = StandardScaler()
    pca    = PCA(n_components=PCA_DIM, whiten=True, random_state=42)

    emb_tr_pca = pca.fit_transform(scaler.fit_transform(emb_full[tr]))
    emb_va_pca = pca.transform(scaler.transform(emb_full[va]))

    X_tr = np.concatenate([emb_tr_pca, scores_raw[tr]], axis=1)
    X_va = np.concatenate([emb_va_pca, scores_raw[va]], axis=1)

    active = [c for c in range(N_CLASSES) if Y[tr, c].sum() >= MIN_POS]
    for c in active:
        clf = LogisticRegression(C=0.5, max_iter=1000, solver="lbfgs")
        clf.fit(X_tr, Y[tr, c])
        oof_preds[va, c] = clf.predict_proba(X_va)[:, 1]

    auc_f, n_f = macro_auc(Y[va], oof_preds[va])
    print(f"  Fold {fold}: AUC={auc_f:.4f} ({n_f} classes, {len(active)} probes)")

auc_oof, n  = macro_auc(Y, oof_preds)
cmap_oof    = padded_cmap(Y, oof_preds)
print(f"\nOOF LR probes  →  Macro AUC: {auc_oof:.4f} ({n} classes)  |  Padded cmAP: {cmap_oof:.4f}")

  Fold 0: AUC=0.4779 (32 classes, 67 probes)


  Fold 1: AUC=0.5377 (36 classes, 64 probes)


  Fold 2: AUC=0.4786 (36 classes, 65 probes)


  Fold 3: AUC=0.5655 (44 classes, 62 probes)


  Fold 4: AUC=0.4451 (35 classes, 64 probes)



OOF LR probes  →  Macro AUC: 0.6177 (75 classes)  |  Padded cmAP: 0.7810


In [7]:
# ── Summary ──────────────────────────────────────────────────────────────────
print("=" * 55)
print("Experiment 01 Results")
print("=" * 55)
print(f"{'Method':<30} {'Macro AUC':>10} {'Padded cmAP':>12}")
print("-" * 55)
print(f"{'Raw Perch logits':<30} {auc_raw:>10.4f} {cmap_raw:>12.4f}")
print(f"{'+ LR probes (OOF)':<30} {auc_oof:>10.4f} {cmap_oof:>12.4f}")
print("=" * 55)

Experiment 01 Results
Method                          Macro AUC  Padded cmAP
-------------------------------------------------------
Raw Perch logits                   0.4869       0.7736
+ LR probes (OOF)                  0.6177       0.7810
